# Лабораторна робота №2 — Частина 1

## VHI-індекс для адміністративних одиниць України

У цій частині роботи виконується завантаження, очищення та аналіз даних VCI/TCI/VHI з NOAA.

## Завдання 1

Створити програмний код для завантаження структурованих файлів з VHI-індексом для адміністративних одиниць України. Файли мають зберігатися з датою та часом завантаження, а повторні запуски скрипту не повинні створювати дублікати.

In [ ]:
from pathlib import Path
from datetime import datetime
import urllib.request
import pandas as pd
import numpy as np

RAW_DATA_DIR = Path("data/raw/vhi")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 1981
YEAR_END = 2024

# Маппінг NOAA-індексів до назв (англ.)
NOAA_PROVINCES = {
    1: "Cherkasy",
    2: "Chernihiv",
    3: "Chernivtsi",
    4: "Crimea",
    5: "Dnipropetrovsk",
    6: "Donetsk",
    7: "Ivano-Frankivsk",
    8: "Kharkiv",
    9: "Kherson",
    10: "Khmelnytskyi",
    11: "Kyiv Oblast",
    12: "Kyiv City",
    13: "Kirovohrad",
    14: "Luhansk",
    15: "Lviv",
    16: "Mykolaiv",
    17: "Odesa",
    18: "Poltava",
    19: "Rivne",
    20: "Sevastopol",
    21: "Sumy",
    22: "Ternopil",
    23: "Zakarpattia",
    24: "Vinnytsia",
    25: "Volyn",
    26: "Zaporizhzhia",
    27: "Zhytomyr",
}

# Нова індексація за українською абеткою (1 - Вінницька, ...)
UKRAINIAN_REGION_INDEX = {
    24: (1, "Вінницька"),
    25: (2, "Волинська"),
    5: (3, "Дніпропетровська"),
    6: (4, "Донецька"),
    27: (5, "Житомирська"),
    23: (6, "Закарпатська"),
    26: (7, "Запорізька"),
    7: (8, "Івано-Франківська"),
    11: (9, "Київська"),
    13: (10, "Кіровоградська"),
    14: (11, "Луганська"),
    15: (12, "Львівська"),
    16: (13, "Миколаївська"),
    17: (14, "Одеська"),
    18: (15, "Полтавська"),
    19: (16, "Рівненська"),
    21: (17, "Сумська"),
    22: (18, "Тернопільська"),
    8: (19, "Харківська"),
    9: (20, "Херсонська"),
    10: (21, "Хмельницька"),
    1: (22, "Черкаська"),
    3: (23, "Чернівецька"),
    2: (24, "Чернігівська"),
    4: (25, "АР Крим"),
    12: (26, "м. Київ"),
    20: (27, "м. Севастополь"),
}

def get_noaa_url(province_id: int) -> str:
    return (
        "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?"
        f"country=UKR&provinceID={province_id}&year1={YEAR_START}&year2={YEAR_END}&type=Mean"
    )

def download_vhi_file(province_id: int, force: bool = False) -> Path:
    """Завантажує файл для області, якщо він ще не завантажений."""
    existing_files = sorted(RAW_DATA_DIR.glob(f"province_{province_id:02d}_*.csv"))
    if existing_files and not force:
        print(f"Province {province_id:02d}: файл уже існує, повторне завантаження пропущено.")
        return existing_files[-1]

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = RAW_DATA_DIR / f"province_{province_id:02d}_{timestamp}.csv"
    url = get_noaa_url(province_id)

    with urllib.request.urlopen(url) as response:
        content = response.read().decode("utf-8", errors="ignore")

    file_path.write_text(content, encoding="utf-8")
    print(f"Province {province_id:02d}: збережено {file_path}")
    return file_path

def download_all_vhi_files(force: bool = False) -> list[Path]:
    files = []
    for province_id in NOAA_PROVINCES:
        files.append(download_vhi_file(province_id, force=force))
    return files

In [ ]:
# Запуск завантаження даних (якщо файли вже є – пропускаємо)
downloaded_files = download_all_vhi_files(force=False)
print(f"Завантажено файлів: {len(downloaded_files)}")

## Завдання 2

Зчитати завантажені текстові файли у pandas DataFrame. Виконати data cleaning: прибрати зайвий текст, привести типи даних, обробити пропуски, додати назву області та новий індекс області.

**Виправлення згідно з правками викладача:**
- Використовуємо `pd.read_csv` для читання (попередньо очистивши текст від HTML-тегів).
- Не заповнюємо пропуски медіаною, а видаляємо рядки з відсутніми значеннями (бо даних багато).
- Обчислюємо VHI як (VCI + TCI) / 2, оскільки в оригінальних файлах він часто відсутній.

In [ ]:
import re
from io import StringIO

def parse_vhi_file(file_path: Path, province_id: int) -> pd.DataFrame:
    """Парсить NOAA-файл у DataFrame."""
    raw_text = file_path.read_text(encoding="utf-8", errors="ignore")
    # Видаляємо HTML-теги
    clean_text = re.sub(r"<[^>]+>", "", raw_text)
    # Знаходимо рядки, що починаються з цифр (дані)
    lines = clean_text.splitlines()
    data_lines = []
    for line in lines:
        line = line.strip()
        if re.match(r"^\d{4}\s*,|^\d{2}\s*,", line):  # рік або тиждень? В оригіналі рік 4 цифри
            data_lines.append(line)
    if not data_lines:
        raise ValueError(f"У файлі {file_path} не знайдено рядків з даними.")

    csv_text = "year,week,SMN,SMT,VCI,TCI,VHI\n" + "\n".join(data_lines)
    # Використовуємо pd.read_csv
    df = pd.read_csv(StringIO(csv_text))
    # Приводимо до числових типів
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    # -1 часто означає відсутнє значення
    df = df.replace(-1, np.nan)
    # Видаляємо рядки, де немає року або тижня (базові поля)
    df = df.dropna(subset=["year", "week"])
    # Обчислюємо VHI, якщо він відсутній (за формулою)
    if "VHI" in df.columns:
        df["VHI"] = df["VHI"].fillna((df["VCI"] + df["TCI"]) / 2)
    else:
        df["VHI"] = (df["VCI"] + df["TCI"]) / 2
    # Видаляємо рядки, де VHI все ще NaN (якщо VCI або TCI були NaN)
    df = df.dropna(subset=["VHI", "VCI", "TCI"])
    
    df["year"] = df["year"].astype(int)
    df["week"] = df["week"].astype(int)

    ua_index, ua_name = UKRAINIAN_REGION_INDEX[province_id]
    df["province_noaa_id"] = province_id
    df["province_noaa_name"] = NOAA_PROVINCES[province_id]
    df["province_ua_index"] = ua_index
    df["province_ua_name"] = ua_name

    return df

def load_all_vhi_data(data_dir: Path = RAW_DATA_DIR) -> pd.DataFrame:
    frames = []
    for province_id in NOAA_PROVINCES:
        files = sorted(data_dir.glob(f"province_{province_id:02d}_*.csv"))
        if not files:
            print(f"Для province {province_id:02d} файл не знайдено.")
            continue
        df = parse_vhi_file(files[-1], province_id)
        frames.append(df)
    if not frames:
        raise ValueError("Не вдалося завантажити жодного файлу з даними.")
    df = pd.concat(frames, ignore_index=True)
    df = df.sort_values(["province_ua_index", "year", "week"]).reset_index(drop=True)
    return df

vhi_df = load_all_vhi_data()
vhi_df.head()

In [ ]:
vhi_df.info()

In [ ]:
vhi_df[["province_ua_index", "province_ua_name", "province_noaa_id", "province_noaa_name"]].drop_duplicates().sort_values("province_ua_index")

## Завдання 3

Реалізувати процедури для формування вибірок: VHI для області за рік, VHI за діапазон років для областей, екстремуми, середнє та медіана.

In [ ]:
def get_vhi_for_region_year(df: pd.DataFrame, region_index: int, year: int) -> pd.DataFrame:
    """Повертає ряд VHI для області за вказаний рік."""
    result = df[(df["province_ua_index"] == region_index) & (df["year"] == year)]
    return result[["province_ua_index", "province_ua_name", "year", "week", "VHI"]].reset_index(drop=True)

def get_vhi_for_regions_year_range(df: pd.DataFrame, region_indices: list[int], start_year: int, end_year: int) -> pd.DataFrame:
    """Повертає VHI для вказаних областей у заданому діапазоні років."""
    result = df[
        (df["province_ua_index"].isin(region_indices))
        & (df["year"].between(start_year, end_year))
    ]
    return result[["province_ua_index", "province_ua_name", "year", "week", "VHI"]].reset_index(drop=True)

def get_vhi_statistics(df: pd.DataFrame, region_indices: list[int], start_year: int, end_year: int) -> pd.DataFrame:
    """Обчислює min, max, mean та median для VHI за областями і роками."""
    filtered = df[
        (df["province_ua_index"].isin(region_indices))
        & (df["year"].between(start_year, end_year))
    ]
    stats = (
        filtered.groupby(["province_ua_index", "province_ua_name"])["VHI"]
        .agg(min_vhi="min", max_vhi="max", mean_vhi="mean", median_vhi="median")
        .reset_index()
    )
    return stats

def get_drought_weeks(df: pd.DataFrame, region_index: int, start_year: int, end_year: int, threshold: float = 15) -> pd.DataFrame:
    """Додаткова вибірка: тижні з VHI нижче заданого порогу."""
    result = df[
        (df["province_ua_index"] == region_index)
        & (df["year"].between(start_year, end_year))
        & (df["VHI"] < threshold)
    ]
    return result[["province_ua_index", "province_ua_name", "year", "week", "VHI"]].reset_index(drop=True)

### Приклад 1

Ряд VHI для Вінницької області за 2020 рік.

In [ ]:
print(get_vhi_for_region_year(vhi_df, region_index=1, year=2020).shape)
get_vhi_for_region_year(vhi_df, region_index=1, year=2020).head()

### Приклад 2

Ряд VHI за 2018–2020 роки для Вінницької, Волинської та Дніпропетровської областей.

In [ ]:
df_range = get_vhi_for_regions_year_range(vhi_df, region_indices=[1, 2, 3], start_year=2018, end_year=2020)
print(df_range.shape)
df_range.head()

### Приклад 3

Пошук екстремумів, середнього та медіани VHI для вказаних областей і років.

In [ ]:
stats = get_vhi_statistics(vhi_df, region_indices=[1, 2, 3], start_year=2018, end_year=2020)
print(stats.shape)
stats

### Приклад 4

Додаткова вибірка: тижні з дуже низьким VHI.

In [ ]:
drought = get_drought_weeks(vhi_df, region_index=1, start_year=2010, end_year=2020, threshold=15)
print(drought.shape)
drought.head()